# 🛠️ dbt (data build tool) — Course 1: Introduction to dbt

> **Platform:** DataCamp | **Instructor:** Mike Metzger (Data Engineer)
> **Level:** Beginner → Intermediate | **Stack:** dbt-core · DuckDB · SQL · YAML · Jinja

---

## 📋 About This Course

This course was my entry point to **dbt**, the tool that owns the **T** of a modern **ELT** pipeline. Starting from an empty folder, I built the `nyc_yellow_taxi` project on top of the **NYC Yellow Taxi trip records (Jan 2023, Parquet)** using **DuckDB** as a local warehouse: I initialized and configured the project, wrote and materialized SQL models, generated project documentation, used Jinja templates and wired models into a dependency graph (DAG) with `ref()`.

This notebook is my write-up of the course: for every chapter I summarize the concepts in my own words and then show the hands-on exercises I solved, with the code I wrote and the screenshots of the terminal output as evidence.

---

## 📚 Table of Contents

| Chapter | Topic | Key commands / concepts |
|---------|-------|-------------------------|
| [Chapter 1](#ch1) | Foundations of dbt | `dbt`, `dbt --version`, `dbt -h`, `dbt init`, `profiles.yml`, `dbt debug` |
| [Chapter 2](#ch2) | dbt projects and models | dbt workflow, `dbt run`, tables vs views, models, `dbt_project.yml`, `model_properties.yml` |
| [Chapter 3](#ch3) | Documentation, Jinja and hierarchies | `dbt docs generate / serve`, Jinja `{{ }}` / `{% %}`, `dbt compile`, `ref()`, DAG |
| [Summary](#summary) | What I can do now | Skills checklist |

> 📚 **Lesson slides:** `Ch1_Foundations_of_dbt.pdf` | `Ch2_dbt_Projects_and_Models.pdf` | `Ch3_Documentation_Jinja_and_Hierarchies.pdf`
> 🖼️ **Evidence:** every screenshot referenced below lives in [`screenshots/`](./screenshots)

---

<a id="ch1"></a>
## 📌 Chapter 1 — Foundations of dbt

> **Goal:** Understand what dbt is, where it fits in a data pipeline, and get a first project created and connected to a warehouse.

---

### 📖 Key concepts

**What dbt is.** dbt (*data build tool*) handles the **Transform** step of ELT: the data is already loaded into the warehouse, and dbt turns raw tables into clean, documented, tested models using plain `SELECT` statements. Because it abstracts the warehouse connection, the same project can be pointed at DuckDB for development and at Snowflake, BigQuery or Postgres for production, and dbt translates between SQL dialects.

**What it gives a data team:**
- Transformations defined as **SQL models** (version-controlled text files)
- **Relationships** between models (e.g. customers → addresses → orders)
- Execution of the whole transformation process in the right order
- **Data quality tests** and **auto-generated documentation**

**Flavours.** `dbt-core` is the open-source CLI (Mac / Windows / Linux) used throughout this course; dbt Cloud is the hosted, proprietary offering. Its main users are Analytics Engineers, Data Engineers and Data Analysts.

**A dbt project** is simply a folder structure holding the project configuration, the sources and destinations, the SQL models, templates and documentation. It is created with `dbt init <project_name>`, which generates the standard sub-folders:

| Folder | Purpose |
|--------|---------|
| `models/` | SQL (or Python) transformations |
| `seeds/` | Small static CSV files to load |
| `snapshots/` | Change tracking (SCD Type 2) |
| `tests/` | Custom data tests |
| `macros/` | Reusable Jinja functions |
| `analyses/` | Ad-hoc analytical SQL |

**Profiles (`profiles.yml`).** A *profile* describes a deployment target (dev, staging/test, prod) and how to connect to it. A project can have several outputs and selects one with `target:`:

```yaml
nyc_yellow_taxi:
  outputs:
    dev:
      type: duckdb
      path: dbt.duckdb
    prod:
      type: snowflake
      # ...
  target: dev
```

**YAML** is indentation-sensitive (like Python), so keeping entries aligned matters. **DuckDB**, the warehouse used here, is an open-source, serverless, vectorized analytical database — similar to SQLite but built for analytics.

### 🧪 Exercise 1.1 — Running the dbt CLI

My first contact with the tool: running `dbt` without arguments in the terminal prints the usage, the global options and the list of available sub-commands.

```bash
dbt
```

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch1_01_dbt_command.png" alt="Output of the bare dbt command in the IDE terminal" width="720">

*Output of the bare dbt command in the IDE terminal*

### 🧪 Exercise 1.2 — Checking the installed version

Before working on a project it is important to know which dbt-core version is installed (features and syntax change between versions).

```bash
dbt --version
```

**Result:** dbt-core **1.8.9** was installed, and the CLI also reported that a newer release (1.12.5) was available.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch1_02_dbt_version.png" alt="dbt --version showing the installed and latest dbt-core versions" width="560">

*dbt --version showing the installed and latest dbt-core versions*

### 🧪 Exercise 1.3 — Exploring the available commands

To discover what dbt can do, I used the help flag, which lists every option (`--debug`, `--fail-fast`, `--defer`, …) and sub-command.

```bash
dbt -h
```

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch1_03_dbt_help.png" alt="dbt -h listing the CLI options and sub-commands" width="620">

*dbt -h listing the CLI options and sub-commands*

### 🧪 Exercise 1.4 — Initializing the `nyc_yellow_taxi` project

**Scenario:** set up a sample project for the NYC Yellow Taxi dataset. The project name and database type were provided in `ProjectInfo.txt` (`nyc_yellow_taxi`, `duckdb`).

```bash
dbt init nyc_yellow_taxi      # create the project (choosing duckdb as the adapter)
cd nyc_yellow_taxi
ls
```

**Result:** the project folder was created with the six standard sub-directories (`analyses`, `macros`, `models`, `seeds`, `snapshots`, `tests`) plus `dbt_project.yml` and a `README.md`.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch1_04_dbt_init.png" alt="Project created by dbt init, with its standard folder structure" width="720">

*Project created by dbt init, with its standard folder structure*

### 🧪 Exercise 1.5 — Creating the project profile and validating the connection

**Scenario:** the project must point to a local DuckDB database. I edited `profiles.yml` so the profile name matches the project and the adapter type is `duckdb`, and then validated the whole configuration:

```yaml
# profiles.yml
nyc_yellow_taxi:
  outputs:
    dev:
      type: duckdb
      path: dbt.duckdb
  target: dev
```

```bash
dbt debug
```

**Result:** `dbt debug` connected to `dbt.duckdb` (schema `main`) and finished with **"Connection test: OK connection ok — All checks passed!"**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch1_05_profiles_dbt_debug.png" alt="dbt debug confirming the DuckDB connection and project configuration" width="720">

*dbt debug confirming the DuckDB connection and project configuration*

> 💡 **Takeaway:** `dbt init` + `profiles.yml` + `dbt debug` is the minimum setup of any dbt project. `dbt debug` is the first command I run whenever a project does not connect.

---

<a id="ch2"></a>
## 📌 Chapter 2 — dbt Projects and Models

> **Dataset:** `yellow_tripdata_2023-01-partial.parquet` — 300,000 NYC Yellow Taxi trips (January 2023)
> **Goal:** Follow the dbt workflow end-to-end: create models, materialize them in the warehouse, and update them to answer a business request.

---

### 📖 Key concepts

**The dbt workflow**

1. Create the project — `dbt init`
2. Define where the data lives — `profiles.yml`
3. Create / use models and templates
4. Materialize the models — `dbt run`
5. Verify, test and troubleshoot
6. Repeat as needed

**`dbt run`** executes every model and *materializes* it, i.e. turns the `SELECT` into a real object in the warehouse. It should be re-run whenever a model changes; its log reports success or failure per model. `dbt run -f` (`--full-refresh`) forces everything to be rebuilt.

**Tables vs Views** — the two basic materializations dbt can create:

| | Table | View |
|---|---|---|
| Stores data | ✅ Yes — takes space | ❌ No — only the query definition |
| When content is computed | When it is (re)built | Every time it is queried |
| Typical use | Heavy / frequently queried results | Light transformations, always up to date |

**Data model vs dbt model.** A *data model* is a conceptual representation of what the data means and how its parts relate; it facilitates collaboration and is not the same as a data structure. A **dbt model** is one concrete transformation: a `.sql` file (Python is also supported in newer versions) containing a `SELECT` statement, stored under `models/`.

**Reading Parquet with DuckDB.** Parquet is a columnar binary format that DuckDB reads natively:

```sql
select * from read_parquet('file.parquet');
select * from 'file.parquet';            -- equivalent shorthand
```

**Configuration files**

| File | Scope | Contains |
|------|-------|----------|
| `dbt_project.yml` | Whole project (exactly one) | Name, version, profile, folder paths, global materialization settings |
| `profiles.yml` | Whole project | Connection targets (dev / prod) |
| `models/*.yml` (e.g. `model_properties.yml`) | Models | Descriptions, documentation, tests, access… — any name, as many files as needed |

**Updating models** follows the usual engineering loop: pull from source control → find the model → change the query → `dbt run` (or `dbt run -f`) → verify → commit.

### 🧪 Exercise 2.1 — Ordering the dbt workflow

A quick check that I understood the end-to-end order of a dbt project, from `dbt init` to verifying the warehouse contents.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch2_01_workflow_order.png" alt="Correct order of the dbt workflow" width="560">

*Correct order of the dbt workflow*

### 🧪 Exercise 2.2 — Running a project and fixing a broken model

**Scenario:** demo the project for my manager by materializing the raw taxi data. The first `dbt run` failed because the source query in `taxi_rides_raw.sql` was wrong.

**Fix applied in `models/taxi_rides/taxi_rides_raw.sql`:**

```sql
with source_data as (
    select * from read_parquet('yellow_tripdata_2023-01-partial.parquet')
)

select * from source_data
```

```bash
dbt run
./datacheck      # validation script provided by the team (not part of dbt)
```

**Result:** the model materialized correctly and the validation confirmed **300,000 total rides** in the warehouse.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch2_02_running_project.png" alt="Corrected taxi_rides_raw model and datacheck showing 300,000 rides" width="720">

*Corrected taxi_rides_raw model and datacheck showing 300,000 rides*

### 🧪 Exercise 2.3 — What is (and isn't) a data model

Classifying statements about data models: they **are** conceptual, **classify common attributes** of data and **facilitate collaboration**; they are **not** the same as a data structure and are **not** always written in SQL.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch2_03_data_model_true_false.png" alt="True / false statements about data models" width="720">

*True / false statements about data models*

### 🧪 Exercise 2.4 — Creating my first dbt model

**Goal:** a simple model exposing every column of the January 2023 Parquet file, using DuckDB's shorthand syntax.

```sql
-- models/taxi_rides/taxi_rides_raw.sql
select * from 'yellow_tripdata_2023-01-partial.parquet'
```

```bash
cd nyc_yellow_taxi
dbt run
```

**Result:** `1 of 1 OK created sql view model` → **PASS=1 WARN=0 ERROR=0**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch2_04_creating_model.png" alt="First dbt model materialized as a view with dbt run" width="720">

*First dbt model materialized as a view with dbt run*

### 🧪 Exercise 2.5 — Project-level vs model-level configuration files

`dbt_project.yml` and `profiles.yml` configure the whole project; `model_properties.yml`, `models/test_model.yml` and the model's own `.sql` file only affect models.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch2_05_config_files.png" alt="Configuration files classified by scope" width="720">

*Configuration files classified by scope*

### 🧪 Exercise 2.6 — Answering a business request: credit-card riders per day

**Scenario:** Marketing asked for the number of trips paid **by credit card** per day, available in the warehouse and refreshed automatically with the rest of the models. Using the dataset's data dictionary I found that `payment_type = 1` means *credit card*.

```sql
-- models/taxi_rides/total_creditcard_riders_by_day.sql
select
    date_part('day', tpep_pickup_datetime) as day,
    count(*)                               as total_cc_riders
from taxi_rides_raw
where payment_type = 1          -- 1 = credit card (data dictionary)
group by day
```

```bash
dbt run
./datacheck
```

**Result:** a new model with one row per day (e.g. day 1 → 54,743 riders, day 3 → 63,678). The handful of rows for days 24, 25 and 31 fall outside the date range covered by the partial file, which points to inconsistent pickup timestamps in the source — exactly the kind of data-quality issue that the tests in Course 2 are designed to catch.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch2_06_creditcard_riders_model.png" alt="total_creditcard_riders_by_day model and its validated output" width="720">

*total_creditcard_riders_by_day model and its validated output*

> 💡 **Takeaway:** a dbt model is just a `SELECT` saved as a file — dbt takes care of creating (or replacing) the view/table in the warehouse, so answering a new business question means adding one `.sql` file and running `dbt run`.

---

<a id="ch3"></a>
## 📌 Chapter 3 — Documentation, Jinja Templates and Model Hierarchies

> **Goal:** Make the project self-documenting, use Jinja to write dynamic SQL, and let dbt build models in dependency order through `ref()`.

---

### 📖 Key concepts

**Why document.** Documentation centralizes knowledge for data consumers: what each model and column means, how to use it, update / SLA details and the lineage of the data. In dbt it lives next to the code, in the `.yml` files:

```yaml
version: 2
models:
  - name: taxi_rides_raw
    description: Yellow Taxi raw data
    access: public
```

From that, dbt builds a documentation website that also shows column data types, data sizes, tests and the **lineage graph (DAG)** automatically.

| Command | What it does |
|---------|--------------|
| `dbt docs generate` | Builds the documentation site from the project (run it after `dbt run`) |
| `dbt docs serve` | Serves it locally on `http://localhost:8080` — development only |

For production, the generated files are copied to a hosting service (dbt Cloud, Amazon S3, Nginx / Apache…).

**Jinja templates.** Jinja is a text templating language used by dbt (and many other tools) to make SQL dynamic:

| Syntax | Meaning |
|--------|---------|
| `{{ ... }}` | Expression / substitution (e.g. `{{ ref('model') }}`, `{{ column }}`) |
| `{% ... %}` | Statement / control flow (`for`, `if`, `test`, `snapshot`…) |

Jinja can perform **substitutions, calculations and loops**. Useful dbt functions include `ref` (reference a model by name), `config` (model configuration) and `docs`. A loop removes repetitive SQL:

```sql
select
{% for column in ['start_date', 'update_date', 'end_date'] %}
    coalesce({{ column }}, '2025-01-01') as {{ column }}{% if not loop.last %},{% endif %}
{% endfor %}
from events
```

`dbt compile` renders the templates into plain SQL under `target/compiled/…`, which is the code actually sent to the warehouse.

**Model hierarchies (DAG).** Models depend on each other. If a model reads from a hard-coded table name, dbt does not know about the dependency and simply builds models **alphabetically**, which can fail. Replacing the table name with `{{ ref('model_name') }}` registers the dependency, so dbt builds a *directed acyclic graph* and always materializes parents before children:

```sql
-- before                                -- after
select first_name, last_name             select first_name, last_name
from taxi_rides_raw                      from {{ ref('taxi_rides_raw') }}
```

### 🧪 Exercise 3.1 — Documentation workflow

Ordering the steps to deliver documentation to the team: document models/tests → `dbt run` → `dbt docs generate` → copy to a hosting service → consume it in the browser.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch3_01_docs_flow.png" alt="Steps to implement and publish dbt documentation" width="560">

*Steps to implement and publish dbt documentation*

### 🧪 Exercise 3.2 — Generating and serving the project documentation

**Scenario:** too much time was going into explaining models and data types to colleagues, so I automated it with dbt docs. I added the model description to `models/model_properties.yml`:

```yaml
version: 2

models:
  - name: taxi_rides_raw
    description: Initial import of the NYC Yellow Taxi trip data from Parquet source
    access: public
```

```bash
dbt docs generate
dbt docs serve        # Ctrl+C to stop the server
```

**Result:** the documentation site was generated and served at `http://localhost:8080` (dbt 1.5.1). The traceback after `Ctrl+C` is expected when the server is interrupted.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch3_02_docs_generate_serve.png" alt="Model description added and documentation served on port 8080" width="720">

*Model description added and documentation served on port 8080*

### 🧪 Exercise 3.3 — Jinja template vs compiled SQL

**Goal:** understand what dbt sends to the database. The model `total_amounts.sql` uses a Jinja `for` loop over four amount columns; after `dbt compile` I compared it with the generated file in `target/compiled/nyc_yellow_taxi/models/taxi_rides/total_amounts.sql`.

**Template (what I write):**
```sql
select
{% for column_name in ['fare_amount', 'tip_amount', 'tolls_amount', 'total_amount'] %}
  {{column_name}}{% if not loop.last %},{% endif %}
{% endfor %}
from taxi_rides_raw
```

**Compiled (what dbt runs):**
```sql
select
  fare_amount,
  tip_amount,
  tolls_amount,
  total_amount
from taxi_rides_raw
```

`loop.last` is what prevents a trailing comma after the last column.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch3_03_jinja_template.png" alt="Jinja template in models/" width="620">

*Jinja template in models/*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch3_04_jinja_compiled.png" alt="Compiled SQL generated in target/compiled/" width="420">

*Compiled SQL generated in target/compiled/*

### 🧪 Exercise 3.4 — Predicting the build order

Given `sales_raw` and three models that read from it via `{{ ref('sales_raw') }}`, dbt builds `sales_raw` first (it is the parent) and then the children — which have no dependency among themselves, so they follow alphabetical order.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch3_05_model_hierarchy_order.png" alt="Order in which dbt materializes the hierarchy" width="560">

*Order in which dbt materializes the hierarchy*

### 🧪 Exercise 3.5 — Fixing a broken hierarchy with `ref()`

**Scenario:** `dbt run` failed because `creditcard_riders_by_day` queried `taxi_rides_raw` by its literal name, so dbt tried to build it before its parent existed. I replaced the table name with a `ref()` and forced a full refresh:

```sql
-- models/taxi_rides/creditcard_riders_by_day.sql
select
    date_part('day', tpep_pickup_datetime) as day,
    count(*)                               as total_riders
from {{ ref('taxi_rides_raw') }}
where payment_type = 1
group by day
```

```bash
dbt run -f
```

**Result:** both models built in the correct order — **PASS=2 WARN=0 ERROR=0 TOTAL=2**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course1_IntroToDbt/screenshots/ch3_06_ref_hierarchy_fix.png" alt="Hierarchy fixed with ref() and full refresh completed successfully" width="720">

*Hierarchy fixed with ref() and full refresh completed successfully*

> 💡 **Takeaway:** `ref()` is the most important function in dbt — it turns a set of SQL files into a dependency graph, which is what enables correct build order, lineage in the docs and selective runs.

---

<a id="summary"></a>
## ✅ Summary — What I Can Do After This Course

| Skill | Evidence in this notebook |
|-------|---------------------------|
| Install-check and explore the dbt CLI | Exercises 1.1 – 1.3 |
| Create a dbt project and configure a warehouse connection (`profiles.yml`, `dbt debug`) | Exercises 1.4 – 1.5 |
| Write SQL models over Parquet data and materialize them with `dbt run` | Exercises 2.2, 2.4 |
| Translate a business request into a new model | Exercise 2.6 |
| Distinguish project vs model configuration files | Exercise 2.5 |
| Document models and generate / serve dbt docs | Exercises 3.1 – 3.2 |
| Write dynamic SQL with Jinja and read the compiled output | Exercise 3.3 |
| Build a dependency graph (DAG) with `ref()` and debug build-order errors | Exercises 3.4 – 3.5 |

**Next step →** [Course 2: Intermediate dbt](../Course2_IntermediateDbt/dbt_Course2_IntermediateDbt.ipynb): data tests, sources, seeds, snapshots (SCD2) and `dbt build` for production.